In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!mkdir -p /content/data
!cp -r "/content/drive/MyDrive/RAVDESS_DATA/ravdess" /content/data/ravdess

In [ ]:
!find "/content/drive/MyDrive/RAVDESS_DATA" -iname "*.csv"

/content/drive/MyDrive/RAVDESS_DATA/metadata.csv


In [ ]:
import pandas as pd
df = pd.read_csv(os.environ["SER_METADATA_CSV"])

In [ ]:
df.head()

,filepath,emotion,actor
0,C:\Users\Sanchit\OneDrive\Desktop\Speech Emoti...,Neutral,1
1,C:\Users\Sanchit\OneDrive\Desktop\Speech Emoti...,Neutral,1
2,C:\Users\Sanchit\OneDrive\Desktop\Speech Emoti...,Neutral,1
3,C:\Users\Sanchit\OneDrive\Desktop\Speech Emoti...,Neutral,1
4,C:\Users\Sanchit\OneDrive\Desktop\Speech Emoti...,Neutral,1


In [ ]:
print(df["filepath"].iloc[0])
print(os.path.exists(df["filepath"].iloc[0]))

C:\Users\Sanchit\OneDrive\Desktop\Speech Emotion Recognition\data\raw\Actor_01\03-01-01-01-01-01-01.wav
False


In [ ]:
import os, shutil
import pandas as pd

DRIVE_METADATA = "/content/drive/MyDrive/RAVDESS_DATA/metadata.csv"  # adjust if it's elsewhere
LOCAL_DATA_DIR = "/content/data/ravdess"
LOCAL_METADATA = "/content/data/metadata.csv"

os.makedirs("/content/data", exist_ok=True)
if not os.path.exists(LOCAL_DATA_DIR):
    print("Copying audio files locally (takes a minute)...")
    shutil.copytree("/content/drive/MyDrive/RAVDESS_DATA/ravdess", LOCAL_DATA_DIR)

df = pd.read_csv(DRIVE_METADATA)

# lowercase emotion labels: "Neutral" -> "neutral"
df["emotion"] = df["emotion"].str.lower()

# rebuild path for Colab using actor id + filename (ignores the old Windows path)
def fix_path(row):
    old = str(row["filepath"]).replace("\\", "/")
    filename = old.split("/")[-1]
    actor_folder = f"Actor_{int(row['actor']):02d}"
    return os.path.join(LOCAL_DATA_DIR, actor_folder, filename)

df["path"] = df.apply(fix_path, axis=1)
df = df.drop(columns=["filepath"])

found = df["path"].apply(os.path.exists)
print("Files found:", found.sum(), "/", len(df))
print(df[~found].head())  # should be empty

df.to_csv(LOCAL_METADATA, index=False)
df.head()

Files found: 1440 / 1440
Empty DataFrame
Columns: [emotion, actor, path]
Index: []


,emotion,actor,path
0,neutral,1,/content/data/ravdess/Actor_01/03-01-01-01-01-...
1,neutral,1,/content/data/ravdess/Actor_01/03-01-01-01-01-...
2,neutral,1,/content/data/ravdess/Actor_01/03-01-01-01-02-...
3,neutral,1,/content/data/ravdess/Actor_01/03-01-01-01-02-...
4,neutral,1,/content/data/ravdess/Actor_01/03-01-02-01-01-...


In [ ]:
!pip install -q transformers accelerate datasets torchaudio librosa soundfile scikit-learn gradio

In [ ]:
import random, numpy as np, torch

SEED = 42
SAMPLING_RATE = 16000
MAX_LENGTH = int(SAMPLING_RATE * 5.0)
MODEL_CHECKPOINT = "facebook/wav2vec2-base-960h"
OUTPUT_DIR = "/content/outputs"
MODEL_DIR = os.path.join(OUTPUT_DIR, "wav2vec2-ser-ravdess")

LABELS = sorted(df["emotion"].unique().tolist())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(LABELS)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(SEED)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
from datasets import Dataset, Audio, ClassLabel
from sklearn.model_selection import train_test_split

actors = df["actor"].unique()
train_actors, temp_actors = train_test_split(actors, test_size=0.2, random_state=SEED)
val_actors, test_actors = train_test_split(temp_actors, test_size=0.5, random_state=SEED)

train_df = df[df["actor"].isin(train_actors)].reset_index(drop=True)
val_df   = df[df["actor"].isin(val_actors)].reset_index(drop=True)
test_df  = df[df["actor"].isin(test_actors)].reset_index(drop=True)
print("train/val/test:", len(train_df), len(val_df), len(test_df))

class_label = ClassLabel(names=LABELS)

def to_dataset(d):
    ds = Dataset.from_pandas(d[["path", "emotion"]].rename(columns={"path": "audio"}), preserve_index=False)
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    ds = ds.map(lambda ex: {"label": class_label.str2int(ex["emotion"])})
    return ds

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

train/val/test: 1140 120 180


Map:   0%|          | 0/1140 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

def preprocess_batch(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(audio_arrays, sampling_rate=SAMPLING_RATE,
                                max_length=MAX_LENGTH, truncation=True, padding="max_length")
    batch["input_values"] = inputs["input_values"]
    return batch

map_kwargs = dict(batched=True, batch_size=8, remove_columns=["audio", "emotion"])
train_ds = train_ds.map(preprocess_batch, **map_kwargs)
val_ds = val_ds.map(preprocess_batch, **map_kwargs)
test_ds = test_ds.map(preprocess_batch, **map_kwargs)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

Map:   0%|          | 0/1140 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Config, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

model_config = Wav2Vec2Config.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(LABELS), label2id=LABEL2ID, id2label=ID2LABEL,
    finetuning_task="audio-classification",
)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, config=model_config)
model.freeze_feature_encoder()

class DataCollatorSER:
    def __call__(self, features):
        input_values = torch.stack([torch.tensor(f["input_values"], dtype=torch.float32) for f in features])
        labels = torch.tensor([f["label"] for f in features], dtype=torch.long)
        return {"input_values": input_values, "labels": labels}

data_collator = DataCollatorSER()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to=["none"],
    seed=SEED,
)

trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
                       data_collator=data_collator, compute_metrics=compute_metrics)
try:
    trainer = Trainer(processing_class=feature_extractor, **trainer_kwargs)
except TypeError:
    trainer = Trainer(tokenizer=feature_extractor, **trainer_kwargs)

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.weight             | UNEXPECTED | 
classifier.bias            | MISSING    | 
classifier.weight          | MISSING    | 
projector.weight           | MISSING    | 
wav2vec2.masked_spec_embed | MISSING    | 
projector.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=LABELS, digits=4))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
trainer.save_model(MODEL_DIR)
feature_extractor.save_pretrained(MODEL_DIR)

shutil.copytree(MODEL_DIR, "/content/drive/MyDrive/wav2vec2-ser-ravdess", dirs_exist_ok=True)
print("Model saved to Drive.")